In [ ]:
# =============================================================================
# CELL 1 -- RUN THE CONFIGURATION-DRIVEN DYNAMIC MAGNETIC SIMULATION
# one physical integration step per substep; no static-equilibrium loop
# =============================================================================

%matplotlib inline
import os
import sys
import time
from pathlib import Path

# Use this package even when the notebook is launched from the repository root.
project_dir = Path.cwd()
if not (project_dir / "utils" / "parameters.py").is_file():
    project_dir = project_dir / "dynamic_magnet_example"
if not (project_dir / "utils" / "parameters.py").is_file():
    raise RuntimeError("Open this notebook from the dynamic_magnet_example folder.")
loaded_utils = sys.modules.get("utils")
if loaded_utils is not None and Path(loaded_utils.__file__).resolve().parent != project_dir / "utils":
    raise RuntimeError("Restart the kernel: another catheter notebook has already imported its own utils package.")
os.chdir(project_dir)
sys.path.insert(0, str(project_dir))

from utils import mag_parameters, main_code, parameters, paths, trajectory

# ----------------------------- settings --------------------------------------
MATTER = "PDMS2"
FLAG = "DynamicMagnet"
MATERIAL_RECIPE = "Matlab"   # Matlab | Trial3 | StaticMagnet
MAG_SET = "Set3"
SHAPE = "Spiral"              # Step | Spiral | Inf | Line | Sine | Squarewave | Triangle | Square | SpiralTilted
FREQ = 0.20
AMP = 0.10                  # external-magnet path amplitude [m]
ALTITUDE = 0.155            # PDMS2 Matlab/mode.m: L + 0.05 [m]
STEP_POSITION = (0.10, 0.10, 0.155)
EXTERNAL_QUATERNION_WXYZ = (1.0, 0.0, 0.0, 0.0)  # Matlab/trajectory.m

STEPS = 500                # physical output intervals
DT = 0.01                  # output interval [s]; total duration = STEPS * DT
SUBSTEPS = 20               # update mechanics AND magnetic wrench every DT/8 = 0.00125 s
N_ELEM = None              # None -> MATLAB num_points (PDMS2: 50)
NUMERICAL_OVERRIDES = {}    # e.g. {"rayleigh_stiffness": 0.01}
SAVE_DATA = True

# ------------------------- 1. parameters -------------------------------------
overrides = dict(NUMERICAL_OVERRIDES)
overrides.update(flag=FLAG, shape=SHAPE, mag_set=MAG_SET,
                 freq=FREQ, amp=AMP, steps=STEPS, dt=DT, substeps=SUBSTEPS)
params = parameters.build(MATTER, n_elem=N_ELEM,
                          recipe=MATERIAL_RECIPE, **overrides)
magnets = mag_parameters.build(MAG_SET)

# ------------------------- 2. external-magnet poses ---------------------------
p_ext, q_ext = trajectory.build(
    SHAPE, params.steps, FREQ, AMP, altitude=ALTITUDE, dt=params.dt,
    step_position=STEP_POSITION,
    external_quaternion_wxyz=EXTERNAL_QUATERNION_WXYZ,
)

print("=" * 78)
print("DYNAMIC MAGNETIC SOFA CATHETER - configuration-driven coupling")
print("=" * 78)
print(params.summary())
print(magnets.summary())
print(trajectory.describe(p_ext, q_ext, params.dt))

# ------------------------- 3. coupled physical-time solve ---------------------
started = time.perf_counter()
res = main_code.run(params, magnets, p_ext, q_ext,
                    verbose_every=max(1, params.steps // 5))
print()
main_code.report(res)
print(f"Simulation elapsed: {time.perf_counter() - started:.2f} s")
print("\nRun cell 2 to save and plot; run cell 3 to create the video.")


In [ ]:
# =============================================================================
# CELL 2 -- SAVE DATA AND PLOT TIP POSITION / MAGNETIC FORCE / TORQUE
# =============================================================================

from utils import visualization

if SAVE_DATA:
    main_code.save(res, paths.dataset(params))
visualization.plot_tip_and_wrench(res, paths.plot(params), show=True)


In [ ]:
# =============================================================================
# CELL 3 -- CREATE THE 3D VIDEO
# Cuboid external magnet + cylindrical tip magnet, red N / blue S.
# No clamped-base marker. Title displays the physical simulation time.
# =============================================================================

from IPython.display import Image, Video, display
import importlib
from utils import visualization

# Pick up visualization.py changes without restarting or losing res.
visualization = importlib.reload(visualization)

VIDEO_STRIDE = 2
VIDEO_FPS = 50     # real-time playback if VIDEO_FPS = 1 / (DT * VIDEO_STRIDE)
VIDEO_DPI = 110
video_path = paths.video(params)

movie = visualization.animate_3d(
    res, video_path, stride=VIDEO_STRIDE, fps=VIDEO_FPS, dpi=VIDEO_DPI,
    elev=50, azim=160,
)

# Embed the actual saved file (MP4, or GIF if ffmpeg is unavailable).
inline_video_path = movie.saved_path
if inline_video_path.lower().endswith(".mp4"):
    display(Video(filename=inline_video_path, embed=True, width=880,
                  html_attributes="controls"))
else:
    display(Image(filename=inline_video_path, width=880))
